new new new

In [0]:
%pip install langchain==0.2.6
%pip install langchain-core==0.2.1
%pip install langchain-community==0.2.6
%pip install langchain-chroma==0.1.2
%pip install langchain-huggingface==0.0.3
%pip install chromadb==0.4.24
%pip install pypdf==4.2.0

In [0]:
import argparse
from langchain_community.vectorstores import Chroma
from langchain.prompts import ChatPromptTemplate
from dotenv import load_dotenv
import os
import shutil
from langchain.document_loaders import PyPDFLoader
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

lets create db with vector embeddings

In [0]:
# Load environment variables. Assumes that project contains .env file with API keys
load_dotenv()

CHROMA_PATH = "/Workspace/Users/jedrzej.brzezicki@pl.nestle.com/langchain-rag-tutorial/data/chroma"

In [0]:
# !pip install sentence-transformers==2.4.0

In [0]:
# %pip install --upgrade langchain-core langchain-community

In [0]:
# pip install -U langchain-huggingface

In [0]:
import shutil
shutil.rmtree(CHROMA_PATH, ignore_errors=True)

In [0]:
file_path = '/dbfs/FileStore/Sports_Essentials_Football_Coaching_Guide_2021.pdf'
loader = PyPDFLoader(file_path)
pages = loader.load()

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Now create the Chroma DB with the new embedding model
db = Chroma.from_documents(
    pages,
    embeddings,
    persist_directory=CHROMA_PATH
)
db.persist()

In [0]:
# %pip install --upgrade langchain

lets declare prompt

In [0]:
# import argparse
# from langchain_community.vectorstores import Chroma
# from langchain.prompts import ChatPromptTemplate

# PROMPT_TEMPLATE = """
# Answer the question based only on the following context:

# {context}

# ---

# Answer the question based on the above context: {question}
# """

PROMPT_TEMPLATE = """You are a football tactics expert. Use only the information provided in the context below to answer the question. If the answer is not in the context, say "I don't have enough information to answer that."

Context:
{context}

Question: {question}

Answer:"""

look for matching context for example of query text

In [0]:
query_text = "What is Football"

# # Search the DB.
results = db.similarity_search_with_relevance_scores(query_text, k=3)
if len(results) == 0:
    print("No results found.")

context_text = "\n\n---\n\n".join([doc.page_content for doc, _score in results])
prompt_template = ChatPromptTemplate.from_template(PROMPT_TEMPLATE)
prompt = prompt_template.format(context=context_text, question=query_text)
print(prompt)

now having those proposals of context lets put it into llm model

In [0]:
from dotenv import load_dotenv
load_dotenv('.env')
import os
hf_token = os.getenv('hf_token')

# Load model directly
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("meta-llama/Meta-Llama-3-8B", token=hf_token)
model = AutoModelForCausalLM.from_pretrained("meta-llama/Meta-Llama-3-8B", token=hf_token)

do it with pipeline

In [0]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from langchain_huggingface import HuggingFacePipeline
from langchain.schema.runnable import RunnablePassthrough
from langchain.schema.output_parser import StrOutputParser

# Create a pipeline
# pipe = pipeline(
#     "text-generation",
#     model=model,
#     tokenizer=tokenizer,
#     max_new_tokens=512,
#     temperature=0.7,
#     top_p=0.95,
#     repetition_penalty=1.15
# )

# advanced pipeline
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=256,      # Reduce this! Was 512
    temperature=0.3,          # Lower = more focused (was 0.7)
    top_p=0.9,               # Lower = less random
    repetition_penalty=1.2,  # Increase to avoid repetition
    do_sample=True,
    return_full_text=False   # IMPORTANT: Don't return the prompt!
)

# Wrap it for LangChain
llm = HuggingFacePipeline(pipeline=pipe)

# Build the chain
retriever = db.as_retriever(search_kwargs={"k": 3})

# Create the prompt
prompt = ChatPromptTemplate.from_template(PROMPT_TEMPLATE)

chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# Use it!
# query = "What is gegenpressing?"
# response = chain.invoke(query)
# print(response)

do it with simple invoke

In [0]:
def ask_question(query_text, show_sources=True):
    """Simple function wrapper - easy to understand AND reusable"""
    results = db.similarity_search(query_text, k=3)
    
    if not results:
        return "No relevant information found."
    
    context_text = "\n\n---\n\n".join([doc.page_content for doc in results])
    formatted_prompt = prompt_template.format(context=context_text, question=query_text)
    response = llm.invoke(formatted_prompt)
    
    if show_sources:
        print("\nSources:")
        for doc in results:
            print(f"- {doc.metadata}")
    
    return response

# Use it
# answer = ask_question("What is gegenpressing?")
# print(answer)

add memory

In [0]:
from langchain.memory import ConversationBufferMemory
from langchain.chains import ConversationalRetrievalChain

# Create memory
memory = ConversationBufferMemory(
    memory_key="chat_history",
    return_messages=True,
    output_key="answer"
)

# Build conversational chain
qa_chain = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=db.as_retriever(search_kwargs={"k": 3}),
    memory=memory,
    return_source_documents=True
)

# Now you can have conversations!
response1 = qa_chain({"question": "What is gegenpressing?"})
response2 = qa_chain({"question": "How is it different from normal pressing?"})  # Remembers context!
response3 = qa_chain({"question": "Which teams use it?"})

In [0]:
response1['answer']

In [0]:
response2['answer']

In [0]:
response3['answer']

In [0]:
# def chat():
#     print("Football Tactics Assistant (type 'quit' to exit)\n")
    
#     while True:
#         question = input("You: ").strip()
        
#         if question.lower() in ['quit', 'exit', 'q']:
#             print("Goodbye!")
#             break
            
#         if not question:
#             continue
        
#         result = qa_chain({"question": question})
        
#         print(f"\nAssistant: {result['answer']}\n")
        
#         # Optionally show sources
#         show_sources = input("Show sources? (y/n): ").lower()
#         if show_sources == 'y':
#             print("\nSources:")
#             for i, doc in enumerate(result['source_documents'], 1):
#                 print(f"{i}. {doc.metadata}")
#             print()

# chat()

In [0]:
# Create a file: test_questions.json
test_questions = [
    {
        "question": "What is gegenpressing?",
        "expected_keywords": ["press", "immediately", "lose possession", "counter"],
        "category": "pressing"
    },
    {
        "question": "What is a 4-3-3 formation?",
        "expected_keywords": ["formation", "four defenders", "three midfielders", "three forwards"],
        "category": "formations"
    },
    {
        "question": "How does tiki-taka work?",
        "expected_keywords": ["possession", "passing", "short", "movement"],
        "category": "possession"
    },
    # Add 10-20 questions
]

### evaluate results

In [0]:
import json
from collections import defaultdict

def evaluate_answer(answer, expected_keywords):
    """Check if answer contains expected concepts"""
    answer_lower = answer.lower()
    print(answer_lower)
    hits = sum(1 for keyword in expected_keywords if keyword.lower() in answer_lower)
    score = hits / len(expected_keywords)
    
    return {
        "keyword_coverage": score,
        "found_keywords": [kw for kw in expected_keywords if kw.lower() in answer_lower],
        "missing_keywords": [kw for kw in expected_keywords if kw.lower() not in answer_lower]
    }

def run_evaluation(test_questions, qa_function):
    """Run evaluation on all test questions"""
    results = []
    
    for test in test_questions:
        question = test["question"]
        print(f"\nEvaluating: {question}")
        
        # Get answer from your system
        # it took too long and also answers were worse
        # answer = qa_chain({f"question": "{question}"})
        # answer = answer['answer']
        answer = ask_question(question)
        
        # Evaluate
        eval_result = evaluate_answer(answer, test["expected_keywords"])
        
        results.append({
            "question": question,
            "answer": answer,
            "category": test["category"],
            **eval_result
        })
        
        print(f"Score: {eval_result['keyword_coverage']:.2%}")
    
    return results

# Your QA function
def answer_question(question):
    results = db.similarity_search(question, k=3, filter={"is_quiz": False})
    if not results:
        return "No relevant information found."
    
    context = "\n\n".join([doc.page_content for doc in results])
    formatted_prompt = f"""..."""  # Your prompt
    response = llm.invoke(formatted_prompt)
    return response

# Run evaluation
eval_results = run_evaluation(test_questions, answer_question)

## We got several issues so far
1) bad answers (resolved partly with changed hyperparameters and better prompt)
2) long time of response
3) bad test results for memmory chain (partly responded as for evaluation we shoouldnt use memory or we should implement refresh if question is not related)
4) quizez in answers (need to filter out quizez)

In [0]:
import time

def timed_qa(question):
    # Time retrieval
    start = time.time()
    results = db.similarity_search(question, k=3)
    retrieval_time = time.time() - start
    
    # Time prompt formatting
    start = time.time()
    context_text = "\n\n---\n\n".join([doc.page_content for doc in results])
    formatted_prompt = prompt_template.format(context=context_text, question=query_text)
    format_time = time.time() - start
    
    # Time LLM generation
    start = time.time()
    answer = llm.invoke(formatted_prompt)
    llm_time = time.time() - start
    
    print(f"Retrieval: {retrieval_time:.2f}s")
    print(f"Format: {format_time:.2f}s")
    print(f"LLM: {llm_time:.2f}s")
    print(f"TOTAL: {retrieval_time + format_time + llm_time:.2f}s")
    
    return answer

timed_qa("What is gegenpressing?")

lets see if we got gpu

In [0]:
# Check if you're actually using GPU
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Current device: {torch.cuda.current_device()}")
    print(f"Device name: {torch.cuda.get_device_name(0)}")

# Make sure model is on GPU
print(f"Model device: {model.device}")

quick win would be limiting number of tokens for LLM

In [0]:
# advanced pipeline
# old one
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=256,      # Reduce this! Was 512
    temperature=0.3,          # Lower = more focused (was 0.7)
    top_p=0.9,               # Lower = less random
    repetition_penalty=1.2,  # Increase to avoid repetition
    do_sample=True,
    return_full_text=False   # IMPORTANT: Don't return the prompt!
)

# new one
# pipe = pipeline(
#     "text-generation",
#     model=model,
#     tokenizer=tokenizer,
#     max_new_tokens=100,  # ← Reduce this!
#     temperature=0.3,
#     do_sample=False,  # ← Greedy = faster
#     pad_token_id=tokenizer.eos_token_id
# )

# Wrap it for LangChain
llm = HuggingFacePipeline(pipeline=pipe)

# Build the chain
retriever = db.as_retriever(search_kwargs={"k": 3})

# Create the prompt
prompt = ChatPromptTemplate.from_template(PROMPT_TEMPLATE)

chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# Use it! ehhh its not much better
# query = "What is gegenpressing?"
# response = chain.invoke(query)
# print(response)

its a little faster but not perfect still.. lets leave it for now

### version of buffer cleaering with two different questions

In [0]:
from langchain.memory import ConversationBufferMemory
import re
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# Initialize
memory = ConversationBufferMemory(
    memory_key="chat_history",
    return_messages=True,
    output_key="answer"
)

last_question = None

def topics_related(question1, question2, threshold=0.6):
    """Check if questions are related (handles follow-ups)"""
    
    if not question1 or not question2:
        return False
    
    q2_lower = question2.lower().strip()
    
    # Quick checks for follow-up patterns
    followup_patterns = [
        r'\b(it|this|that|they|them)\b',  # Pronouns
        r'^(what about|how about|more)',  # Common starts
        r'\b(more|else|also|detail|elaborate)\b'  # Follow-up words
    ]
    
    for pattern in followup_patterns:
        if re.search(pattern, q2_lower):
            return True
    
    # Check if very short (likely follow-up)
    if len(q2_lower.split()) <= 2:
        return True
    
    # Use embeddings for longer questions
    emb1 = embeddings.embed_query(question1)
    emb2 = embeddings.embed_query(question2)
    similarity = cosine_similarity(
        np.array(emb1).reshape(1, -1),
        np.array(emb2).reshape(1, -1)
    )[0][0]
    print(similarity)
    return similarity > threshold

def answer_with_smart_memory(question):
    global last_question
    
    if last_question:
        if topics_related(last_question, question):
            print("💬 Continuing conversation (memory kept)")
        else:
            print("🔄 New topic detected (memory cleared)")
            memory.clear()
    
    result = qa_chain({"question": question})
    last_question = question
    
    return result

# Test it!
# print("Q1:", answer_with_smart_memory("What is gegenpressing?"))
# print("\nQ2:", answer_with_smart_memory("Tell me more"))  # Should keep memory
# print("\nQ3:", answer_with_smart_memory("Explain it in detail"))  # Should keep memory
# print("\nQ4:", answer_with_smart_memory("What is tiki-taka?"))  # Should clear memory
# These should all return True (keep memory)
print(topics_related("What is gegenpressing?", "Tell me more"))
print(topics_related("What is gegenpressing?", "Explain it"))
print(topics_related("What is gegenpressing?", "What about that?"))
print(topics_related("What is gegenpressing?", "Give me more details"))
print(topics_related("What is gegenpressing?", "How does it work?"))
print(topics_related("What is gegenpressing?", "Why?"))

# These should return False (clear memory)
print(topics_related("What is gegenpressing?", "What is tiki-taka?"))
print(topics_related("What is gegenpressing?", "Explain 4-3-3 formation"))

make test once again

In [0]:
def run_evaluation(test_questions):
    """Run evaluation on all test questions"""
    results = []
    
    for test in test_questions:
        question = test["question"]
        print(f"\nEvaluating: {question}")
        
        # Get answer from your system
        # it took too long and also answers were worse
        answer = answer_with_smart_memory(question)
        answer = answer['answer']
        # answer = ask_question(question)
        
        # Evaluate
        eval_result = evaluate_answer(answer, test["expected_keywords"])
        
        results.append({
            "question": question,
            "answer": answer,
            "category": test["category"],
            **eval_result
        })
        
        print(f"Score: {eval_result['keyword_coverage']:.2%}")
    
    return results

# Run evaluation
eval_results = run_evaluation(test_questions)

In [0]:
eval_results

lets add agents

In [0]:
# %pip install --upgrade langchain-core langchain-community transformers accelerate

In [0]:
pip install --upgrade transformers

In [0]:
dbutils.library.restartPython()

In [0]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "Qwen/Qwen3-Coder-30B-A3B-Instruct"

# load the tokenizer and the model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

# prepare the model input
prompt = "Write a quick sort algorithm."
messages = [
    {"role": "user", "content": prompt}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

# conduct text completion
generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=65536
)
output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist() 

content = tokenizer.decode(output_ids, skip_special_tokens=True)

print("content:", content)


In [0]:
from langchain.agents import AgentExecutor, create_react_agent
from langchain.prompts import PromptTemplate
from langchain.agents import Tool

# Tools
def search_tactics(query: str) -> str:
    print(f"🔍 Searching for: {query}")
    results = db.similarity_search(query, k=3, filter={"is_quiz": False})
    if not results:
        return "No results found."
    return "\n".join([doc.page_content[:200] for doc in results])

tools = [
    Tool(
        name="search_tactics",
        func=search_tactics,
        description="Useful for finding information about football tactics, pressing, formations, and strategies. Input should be a search query."
    )
]

# Simpler ReAct prompt
react_prompt = PromptTemplate.from_template("""
Answer the question using the available tools.

Tools:
{tools}

Tool Names: {tool_names}

Use this format:

Question: the input question
Thought: what should I do?
Action: search_tactics
Action Input: search query
Observation: result from tool
Thought: I now know the answer
Final Answer: the final answer

Question: {input}

{agent_scratchpad}
""")

# Create agent
agent = create_react_agent(llm, tools, react_prompt)
executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    max_iterations=3,
    handle_parsing_errors=True,
    return_intermediate_steps=True
)

# Test it
result = executor.invoke({"input": "What is gegenpressing?"})
print("\n" + "="*50)
print("ANSWER:", result['output'])
print("="*50)

In [0]:
jk

my tools 